# ImageEval 2026 — Task 1a (Spoken VQA) · Colab baseline

End-to-end baseline for **Task 1a**: given an **image** and a **spoken** question
with three options (one `.wav`), predict the correct option index **0, 1, or 2**.

**Model — a true omni model.** [`Qwen2.5-Omni-3B`](https://huggingface.co/Qwen/Qwen2.5-Omni-3B)
ingests the **audio and the image together** (no separate ASR step) and answers directly.
We load it with the speech decoder disabled (`enable_audio_output=False`) since we only need
text out — that keeps it within a **free Colab T4 (16 GB)**.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` (GPU, **not** TPU). If you have
> Colab Pro, an **L4 / A100** is noticeably faster. If a T4 OOMs, lower `MAX_ITEMS` or
> switch to a smaller model — see the note in the load cell.

**Output:** `predictions_<lang>.csv` (`id,raw_prediction,prediction_parsed`) and a
Codabench-ready `prediction_<lang>.zip`. Switch tracks with `LANG = "en"`/`"msa"`.
Defaults to `devtest` (blind, for submission); set `SPLIT="dev"` to get a local score.

## 1. Install dependencies

Installs once, then **auto-restarts the runtime** so the freshly installed versions load
cleanly (this avoids `PIL`/`transformers` half-upgrade import errors). The restart only
happens on the **first** run — when it does, just press **Run all** again to continue.

In [ ]:
# Install once, then restart so new versions load cleanly. Re-run "Run all" after the restart.
import os
_FLAG = "/content/.imageeval_deps_1a"
if not os.path.exists(_FLAG):
    !pip install -q -U "transformers>=4.52.0" accelerate bitsandbytes qwen-omni-utils "huggingface_hub[hf_transfer]" soundfile librosa
    open(_FLAG, "w").close()
    print("Dependencies installed — restarting runtime. Press 'Run all' again to continue.")
    import IPython; IPython.Application.instance().kernel.do_shutdown(True)

## 2. Configuration

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # fast Rust downloader
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less GPU fragmentation

REPO_ID   = "QCRI/AynVQA-ArabicNLP26"
TASK      = "task1a"
LANG      = "en"        # "en" or "msa"  -> output is predictions_<LANG>.csv
SPLIT     = "devtest"   # "devtest"/"test" -> blind (submit) | "dev"/"train" -> labelled (scored)
MAX_ITEMS = None        # e.g. 20 for a quick smoke test; None = whole split

# A true omni model: takes the spoken question/options (.wav) + the image directly.
VLM_MODEL = "Qwen/Qwen2.5-Omni-3B"   # 7B variant: "Qwen/Qwen2.5-Omni-7B"
QUANTIZE  = True       # 4-bit (bitsandbytes) -> fits a T4 comfortably; set False on L4/A100
print(f"config: {TASK}_{LANG} / {SPLIT}  (quantized: {QUANTIZE})")

## 3. Download the split + its media from the Hub

We pull only the JSONL for the chosen split and the exact images/audio it references
(cached by `huggingface_hub`), not the whole dataset.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f"{TASK}/{SPLIT}_{LANG}.jsonl", repo_type="dataset")
records = [json.loads(l) for l in open(jsonl, encoding="utf-8") if l.strip()]
if MAX_ITEMS:
    records = records[:MAX_ITEMS]
print(len(records), "items;  labelled:", "label" in records[0])

def fetch(rel):
    return hf_hub_download(REPO_ID, filename=rel, repo_type="dataset")

needed = sorted({r["image"] for r in records} | {r["audio"] for r in records})
paths = {}
with ThreadPoolExecutor(max_workers=16) as ex:
    for rel, p in tqdm(zip(needed, ex.map(fetch, needed)), total=len(needed), desc="media"):
        paths[rel] = p

## 4. Load the omni model

`enable_audio_output=False` drops the speech-generation head (~2 GB) — we only need text.
On Ampere+ (`L4`/`A100`) it uses bf16; on a T4 it falls back to fp16.

*Memory-tight on a free T4?* This 3B omni model fits but leaves little headroom. If you OOM,
restart the runtime and run a smaller batch (`MAX_ITEMS`), or fall back to the ASR+VLM recipe.

In [ ]:
import torch
from transformers import (Qwen2_5OmniForConditionalGeneration,
                          Qwen2_5OmniProcessor, BitsAndBytesConfig)
from qwen_omni_utils import process_mm_info

# Fail fast on a CPU runtime: this model is unusably slow without a GPU.
assert torch.cuda.is_available(), \
    "No GPU detected. Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

QUANTIZE = globals().get("QUANTIZE", True)   # falls back to True if cell 2 was skipped
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype:", dtype, "| quantized:", QUANTIZE)

# 4-bit NF4 quantization (~halves weight memory) so the model fits a 16 GB T4.
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
) if QUANTIZE else None

# Cap image resolution -> bounds vision tokens (attention memory grows with seq-len^2).
# 512*28*28 keeps a 16 GB T4 happy; raise toward 1280*28*28 on an L4/A100 for more detail.
MAX_PIXELS = 512 * 28 * 28
processor = Qwen2_5OmniProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype, device_map="auto", quantization_config=quant,
    enable_audio_output=False,        # text-only -> skip the speech decoder, saves VRAM
).eval()

## 5. Inference helpers

In [ ]:
import re

# Qwen2.5-Omni works best with its own system identity; we extend it with the task role.
SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable "
          "of perceiving auditory and visual inputs, as well as generating text and speech. "
          "You answer multiple-choice questions about an image by listening to a spoken "
          "question and its options and reasoning about what the image actually shows.")
PROMPT = ("Look at the image and listen to the audio: it is a question about this image, "
          "followed by three answer options read aloud in order (the first option is 0, "
          "the second is 1, the third is 2). Choose the one option that correctly answers "
          "the question for THIS image. Respond with only a single digit: 0, 1, or 2.")

@torch.no_grad()
def predict(image_path, audio_path):
    conv = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "audio", "audio": audio_path},
            {"type": "text",  "text":  PROMPT},
        ]},
    ]
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to(model.device).to(model.dtype)   # casts float tensors only
    gen = model.generate(**inputs, return_audio=False, do_sample=False, max_new_tokens=8)
    out = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)[0]
    del inputs, gen
    torch.cuda.empty_cache()          # release per-item activations -> avoids slow OOM
    return out.strip()

DEFAULT_INDEX = 0   # fallback when the model output has no 0/1/2 (see notebook note)

def parse_index(txt):
    """Return (parsed_index, ok). ok=False means we used the fallback."""
    m = re.search(r"[012]", txt)
    return (int(m.group()), True) if m else (DEFAULT_INDEX, False)

## 6. Run inference

We keep the model's **raw** text and the **parsed** index. If a reply has no `0/1/2`, we fall
back to `DEFAULT_INDEX` (a blank would be scored *wrong*, so a guess is strictly better) and
report how often that happened — inspect `raw_prediction` to judge real model behaviour.

In [ ]:
rows = []   # (id, raw_prediction, prediction_parsed)
n_fallback = 0
for r in tqdm(records, desc="infer"):
    raw = predict(paths[r["image"]], paths[r["audio"]])
    idx, ok = parse_index(raw)
    n_fallback += (not ok)
    rows.append((r["id"], raw, idx))

print(f"done: {len(rows)} predictions  |  unparseable -> defaulted to {DEFAULT_INDEX}: {n_fallback}")

## 7. Write `predictions_<lang>.csv`

Columns: `id, raw_prediction, prediction_parsed`. Switching `LANG` writes a separate file
(`predictions_en.csv` / `predictions_msa.csv`), so the two tracks never overwrite each other.

In [ ]:
import csv
OUT_CSV = f"predictions_{LANG}.csv"
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "raw_prediction", "prediction_parsed"])
    for iid, raw, idx in rows:
        w.writerow([iid, raw, idx])
print("wrote", OUT_CSV, ":", len(rows), "rows")

## 8. Score (official metric: **accuracy**)

Runs only when the split is labelled (`dev`/`train`). `devtest`/`test` are blind.

In [ ]:
gold = {r["id"]: r["label"] for r in records if "label" in r}
if gold:
    pred = {iid: idx for iid, _, idx in rows}
    correct = sum(1 for i in gold if pred.get(i) == gold[i])
    print(f"accuracy on {SPLIT}: {correct/len(gold):.4f}  ({correct}/{len(gold)})")
else:
    print(f"'{SPLIT}' is blind (no labels) — submit to Codabench to get the score.")

## 9. Build the Codabench submission

The leaderboard expects a zip containing a `prediction.csv` with exactly `id,prediction`
(the **parsed** index). We build that from `prediction_parsed` — the raw column stays in your
local file for auditing only. Submit `prediction_<lang>.zip` to the matching competition:
[task1a_en](https://www.codabench.org/competitions/17049/) ·
[task1a_msa](https://www.codabench.org/competitions/17048/).

In [ ]:
import zipfile
with open("prediction.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "prediction"])
    for iid, _, idx in rows:
        w.writerow([iid, idx])
zip_name = f"prediction_{LANG}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv", "prediction.csv")
print("wrote", zip_name, " -> submit this file to Codabench")